# Reproduction notebook: 21_itransformer_strong_baseline_reproduction

This notebook is retained as an executable provenance record for the anonymous supplementary package. Saved outputs and internal development notes have been removed.


In [ ]:

from pathlib import Path
from types import SimpleNamespace
import gc
import importlib
import os
import random
import subprocess
import sys
import time
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 220)
pd.set_option("display.width", 460)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

DATASETS = [
    "ETTm1",
    "ETTh1",
    "Weather",
    "Electricity",
]

HORIZONS = [
    96,
    192,
    336,
    720,
]

TASKS = [
    (dataset, horizon)
    for dataset in DATASETS
    for horizon in HORIZONS
]

SEQ_LEN = 96
LABEL_LEN = 48
SEED = 2023

ROOT = Path(
    "/data/dataset/strong_forecaster/"
    "itransformer_official_baseline_reproduction"
)

CHECKPOINT_DIR = ROOT / "checkpoints"
HISTORY_DIR = ROOT / "history"
ARTIFACT_DIR = ROOT / "artifacts"

for p in [
    ROOT,
    CHECKPOINT_DIR,
    HISTORY_DIR,
    ARTIFACT_DIR,
]:
    p.mkdir(
        parents=True,
        exist_ok=True,
    )

RESUME = True
FORCE_RETRAIN = False

print("Device:", DEVICE)
print("PyTorch:", torch.__version__)
print("Tasks:", len(TASKS))
print("Output:", ROOT)


In [ ]:

REPO_CANDIDATES = [
    Path(
        "/code/stock_regime_retrieval/"
        "strong_forecaster/iTransformer_official"
    ),
    Path("/code/iTransformer"),
    Path("/data/iTransformer"),
]

REPO = next(
    (
        p
        for p in REPO_CANDIDATES
        if (
            p / "model" / "iTransformer.py"
        ).is_file()
        and (
            p / "data_provider" / "data_factory.py"
        ).is_file()
    ),
    None,
)

if REPO is None:
    target = REPO_CANDIDATES[0]

    target.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    print(
        "Official iTransformer repository not found."
    )
    print(
        "Attempting to clone into:",
        target,
    )

    try:
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "https://github.com/thuml/iTransformer.git",
                str(target),
            ],
            check=True,
        )
    except Exception as e:
        raise RuntimeError(
            "Could not clone the official iTransformer repository. "
            "Clone https://github.com/thuml/iTransformer.git manually "
            f"into {target} and rerun this cell."
        ) from e

    REPO = target

if not (
    REPO / "model" / "iTransformer.py"
).is_file():
    raise FileNotFoundError(
        f"Official iTransformer model not found under {REPO}"
    )

try:
    commit = subprocess.check_output(
        [
            "git",
            "-C",
            str(REPO),
            "rev-parse",
            "HEAD",
        ],
        text=True,
    ).strip()
except Exception:
    commit = "unknown"

print("Official repo:", REPO)
print("Commit:", commit)

(ARTIFACT_DIR / "official_repo_commit.txt").write_text(
    commit + "\n"
)


In [ ]:

# Remove potentially conflicting top-level modules.
for module_name in list(sys.modules.keys()):
    if (
        module_name == "model"
        or module_name.startswith("model.")
        or module_name == "layers"
        or module_name.startswith("layers.")
        or module_name == "data_provider"
        or module_name.startswith("data_provider.")
        or module_name == "utils"
        or module_name.startswith("utils.")
    ):
        del sys.modules[module_name]

if str(REPO) in sys.path:
    sys.path.remove(
        str(REPO)
    )

sys.path.insert(
    0,
    str(REPO),
)

itransformer_module = importlib.import_module(
    "model.iTransformer"
)

data_factory_module = importlib.import_module(
    "data_provider.data_factory"
)

tools_module = importlib.import_module(
    "utils.tools"
)

OfficialITransformer = itransformer_module.Model
official_data_provider = data_factory_module.data_provider
official_adjust_lr = tools_module.adjust_learning_rate

actual_model_file = Path(
    itransformer_module.__file__
).resolve()

expected_model_file = (
    REPO / "model" / "iTransformer.py"
).resolve()

print("Imported model:", actual_model_file)

if actual_model_file != expected_model_file:
    raise RuntimeError(
        "Wrong iTransformer implementation imported.\n"
        f"Expected: {expected_model_file}\n"
        f"Actual:   {actual_model_file}"
    )

print(
    "PASS: official iTransformer implementation is active."
)


In [ ]:

DATA_PATH_CANDIDATES = {
    "ETTm1": [
        Path("/data/dataset/ETTm1.csv"),
        Path(
            "/data/Time-Series-Library/"
            "dataset/ETT-small/ETTm1.csv"
        ),
        Path(
            "/data/Time-Series-Library_v2/"
            "dataset/ETT-small/ETTm1.csv"
        ),
    ],
    "ETTh1": [
        Path("/data/dataset/ETTh1.csv"),
        Path(
            "/data/Time-Series-Library/"
            "dataset/ETT-small/ETTh1.csv"
        ),
        Path(
            "/data/Time-Series-Library_v2/"
            "dataset/ETT-small/ETTh1.csv"
        ),
    ],
    "Weather": [
        Path("/data/dataset/weather.csv"),
        Path("/data/dataset/weather/weather.csv"),
        Path(
            "/data/Time-Series-Library/"
            "dataset/weather/weather.csv"
        ),
        Path(
            "/data/Time-Series-Library_v2/"
            "dataset/weather/weather.csv"
        ),
    ],
    "Electricity": [
        Path("/data/dataset/electricity.csv"),
        Path(
            "/data/dataset/electricity/electricity.csv"
        ),
        Path(
            "/data/Time-Series-Library/"
            "dataset/electricity/electricity.csv"
        ),
        Path(
            "/data/Time-Series-Library_v2/"
            "dataset/electricity/electricity.csv"
        ),
    ],
}

DATA_PATHS = {}

for name, candidates in DATA_PATH_CANDIDATES.items():
    hit = next(
        (
            p
            for p in candidates
            if p.is_file()
        ),
        None,
    )

    DATA_PATHS[name] = hit

    print(
        f"{name:11s}:",
        hit if hit is not None else "NOT FOUND",
    )

missing = [
    name
    for name, path in DATA_PATHS.items()
    if path is None
]

if missing:
    print("\nAttempted paths:")

    for name in missing:
        print(f"\n{name}")

        for p in DATA_PATH_CANDIDATES[name]:
            print(" -", p)

    raise FileNotFoundError(
        "Dataset file(s) not found: "
        + ", ".join(missing)
    )


In [ ]:

RECIPES = {
    "ETTm1": {
        "data": "ETTm1",
        "enc_in": 7,
        "e_layers": 2,
        "d_model": 512,
        "d_ff": 2048,
        "n_heads": 8,
        "dropout": 0.1,
        "batch_size": 32,
        "learning_rate": 1e-4,
        "train_epochs": 10,
        "patience": 3,
        "lradj": "type1",
        "factor": 1,
        "freq": "t",
    },
    "ETTh1": {
        "data": "ETTh1",
        "enc_in": 7,
        "e_layers": 2,
        "d_model": 512,
        "d_ff": 2048,
        "n_heads": 8,
        "dropout": 0.1,
        "batch_size": 32,
        "learning_rate": 1e-4,
        "train_epochs": 10,
        "patience": 3,
        "lradj": "type1",
        "factor": 1,
        "freq": "h",
    },
    "Weather": {
        "data": "custom",
        "enc_in": 21,
        "e_layers": 3,
        "d_model": 512,
        "d_ff": 512,
        "n_heads": 8,
        "dropout": 0.1,
        "batch_size": 32,
        "learning_rate": 1e-4,
        "train_epochs": 10,
        "patience": 3,
        "lradj": "type1",
        "factor": 1,
        "freq": "h",
    },
    "Electricity": {
        "data": "custom",
        "enc_in": 321,
        "e_layers": 3,
        "d_model": 512,
        "d_ff": 512,
        "n_heads": 8,
        "dropout": 0.1,
        "batch_size": 16,
        "learning_rate": 5e-4,
        "train_epochs": 10,
        "patience": 3,
        "lradj": "type1",
        "factor": 1,
        "freq": "h",
    },
}

display(
    pd.DataFrame(RECIPES).T
)


In [ ]:

REFERENCE_MSE = {
    ("ETTm1", 96): 0.334,
    ("ETTm1", 192): 0.377,
    ("ETTm1", 336): 0.426,
    ("ETTm1", 720): 0.491,

    ("ETTh1", 96): 0.386,
    ("ETTh1", 192): 0.441,
    ("ETTh1", 336): 0.487,
    ("ETTh1", 720): 0.503,

    ("Weather", 96): 0.174,
    ("Weather", 192): 0.221,
    ("Weather", 336): 0.278,
    ("Weather", 720): 0.358,

    ("Electricity", 96): 0.148,
    ("Electricity", 192): 0.162,
    ("Electricity", 336): 0.178,
    ("Electricity", 720): 0.225,
}


In [ ]:

def make_args(
    name,
    horizon,
):
    r = RECIPES[name]
    path = DATA_PATHS[name]

    return SimpleNamespace(
        # basic
        is_training=1,
        model_id=f"{name}_96_{horizon}",
        model="iTransformer",

        # data
        data=r["data"],
        root_path=str(path.parent) + "/",
        data_path=path.name,
        features="M",
        target="OT",
        freq=r["freq"],
        checkpoints=str(CHECKPOINT_DIR),

        # forecasting
        seq_len=SEQ_LEN,
        label_len=LABEL_LEN,
        pred_len=int(horizon),

        # model
        enc_in=r["enc_in"],
        dec_in=r["enc_in"],
        c_out=r["enc_in"],
        d_model=r["d_model"],
        n_heads=r["n_heads"],
        e_layers=r["e_layers"],
        d_layers=1,
        d_ff=r["d_ff"],
        moving_avg=25,
        factor=r["factor"],
        distil=True,
        dropout=r["dropout"],
        embed="timeF",
        activation="gelu",
        output_attention=False,
        do_predict=False,

        # optimization
        num_workers=10,
        itr=1,
        train_epochs=r["train_epochs"],
        batch_size=r["batch_size"],
        patience=r["patience"],
        learning_rate=r["learning_rate"],
        des="Exp",
        loss="MSE",
        lradj=r["lradj"],
        use_amp=False,

        # GPU
        use_gpu=torch.cuda.is_available(),
        gpu=0,
        use_multi_gpu=False,
        devices="0",

        # iTransformer
        exp_name="MTSF",
        channel_independence=False,
        inverse=False,
        class_strategy="projection",
        target_root_path="./data/electricity/",
        target_data_path="electricity.csv",
        efficient_training=False,
        use_norm=1,
        partial_start_index=0,
    )


In [ ]:

protocol_rows = []

for name, horizon in TASKS:
    args = make_args(
        name,
        horizon,
    )

    train_data, train_loader = official_data_provider(
        args,
        "train",
    )

    val_data, val_loader = official_data_provider(
        args,
        "val",
    )

    test_data, test_loader = official_data_provider(
        args,
        "test",
    )

    protocol_rows.append({
        "Dataset": name,
        "Horizon": horizon,
        "TrainWindows": len(train_data),
        "ValWindows": len(val_data),
        "TestWindows": len(test_data),
        "TrainBatches": len(train_loader),
        "ValBatches": len(val_loader),
        "TestBatches": len(test_loader),
        "TrainBatchSize": args.batch_size,
        "TestBatchSize": 1,
    })

    del (
        train_data,
        train_loader,
        val_data,
        val_loader,
        test_data,
        test_loader,
    )

protocol_df = pd.DataFrame(protocol_rows)

display(protocol_df)

protocol_df.to_csv(
    ARTIFACT_DIR / "data_protocol.csv",
    index=False,
)


In [ ]:

def build_model(
    name,
    horizon,
):
    args = make_args(
        name,
        horizon,
    )

    model = OfficialITransformer(
        args
    ).float().to(
        DEVICE
    )

    return (
        model,
        args,
    )


param_rows = []

for name in DATASETS:
    model, args = build_model(
        name,
        96,
    )

    params = sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )

    param_rows.append({
        "Dataset": name,
        "Params": params,
        "Params_M": params / 1e6,
        "Layers": args.e_layers,
        "d_model": args.d_model,
        "d_ff": args.d_ff,
        "Heads": args.n_heads,
    })

    del model

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

display(
    pd.DataFrame(param_rows)
)


In [ ]:

def set_seed(
    seed=SEED,
):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.benchmark = False


def checkpoint_path(
    name,
    horizon,
):
    return (
        CHECKPOINT_DIR
        / f"{name}_H{horizon}_seed{SEED}.pt"
    )


def history_path(
    name,
    horizon,
):
    return (
        HISTORY_DIR
        / f"{name}_H{horizon}_seed{SEED}.csv"
    )


@torch.no_grad()
def evaluate_loader(
    model,
    loader,
    args,
):
    model.eval()

    batch_mse = []

    sse = 0.0
    sae = 0.0
    count = 0
    windows = 0

    for (
        batch_x,
        batch_y,
        batch_x_mark,
        batch_y_mark,
    ) in loader:
        batch_x = batch_x.float().to(
            DEVICE
        )

        batch_y = batch_y.float().to(
            DEVICE
        )

        if (
            "PEMS" in args.data
            or "Solar" in args.data
        ):
            batch_x_mark = None
            batch_y_mark = None
        else:
            batch_x_mark = batch_x_mark.float().to(
                DEVICE
            )

            batch_y_mark = batch_y_mark.float().to(
                DEVICE
            )

        dec_inp = torch.zeros_like(
            batch_y[
                :,
                -args.pred_len:,
                :
            ]
        ).float()

        dec_inp = torch.cat(
            [
                batch_y[
                    :,
                    :args.label_len,
                    :
                ],
                dec_inp,
            ],
            dim=1,
        ).float().to(
            DEVICE
        )

        outputs = model(
            batch_x,
            batch_x_mark,
            dec_inp,
            batch_y_mark,
        )

        outputs = outputs[
            :,
            -args.pred_len:,
            :
        ]

        true = batch_y[
            :,
            -args.pred_len:,
            :
        ]

        err = outputs - true

        batch_mse.append(
            float(
                (err ** 2).mean().item()
            )
        )

        sse += float(
            (err ** 2).sum().item()
        )

        sae += float(
            err.abs().sum().item()
        )

        count += err.numel()
        windows += len(batch_x)

        del (
            batch_x,
            batch_y,
            batch_x_mark,
            batch_y_mark,
            dec_inp,
            outputs,
            true,
            err,
        )

    return {
        "BatchAverageMSE": float(
            np.mean(batch_mse)
        ),
        "MSE": sse / count,
        "MAE": sae / count,
        "Windows": windows,
        "Batches": len(batch_mse),
        "Elements": count,
    }


In [ ]:

def train_or_load(
    name,
    horizon,
):
    path = checkpoint_path(
        name,
        horizon,
    )

    model, args = build_model(
        name,
        horizon,
    )

    if (
        path.exists()
        and RESUME
        and not FORCE_RETRAIN
    ):
        ckpt = torch.load(
            path,
            map_location=DEVICE,
        )

        model.load_state_dict(
            ckpt["StateDict"]
        )

        model.eval()

        print(
            f"Loaded {name} H={horizon}: "
            f"best={ckpt['BestValMSE']:.6f}"
            f"@{ckpt['BestEpoch']}"
        )

        return (
            model,
            args,
            ckpt,
        )

    set_seed(
        SEED
    )

    train_data, train_loader = official_data_provider(
        args,
        "train",
    )

    val_data, val_loader = official_data_provider(
        args,
        "val",
    )

    model.train()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=args.learning_rate,
    )

    best_val = float("inf")
    best_epoch = -1
    best_state = None
    wait = 0

    history = []

    for epoch in range(
        1,
        args.train_epochs + 1,
    ):
        model.train()

        losses = []
        t0 = time.time()

        for (
            batch_x,
            batch_y,
            batch_x_mark,
            batch_y_mark,
        ) in train_loader:
            optimizer.zero_grad(
                set_to_none=True
            )

            batch_x = batch_x.float().to(
                DEVICE
            )

            batch_y = batch_y.float().to(
                DEVICE
            )

            if (
                "PEMS" in args.data
                or "Solar" in args.data
            ):
                batch_x_mark = None
                batch_y_mark = None
            else:
                batch_x_mark = batch_x_mark.float().to(
                    DEVICE
                )

                batch_y_mark = batch_y_mark.float().to(
                    DEVICE
                )

            dec_inp = torch.zeros_like(
                batch_y[
                    :,
                    -args.pred_len:,
                    :
                ]
            ).float()

            dec_inp = torch.cat(
                [
                    batch_y[
                        :,
                        :args.label_len,
                        :
                    ],
                    dec_inp,
                ],
                dim=1,
            ).float().to(
                DEVICE
            )

            outputs = model(
                batch_x,
                batch_x_mark,
                dec_inp,
                batch_y_mark,
            )

            outputs = outputs[
                :,
                -args.pred_len:,
                :
            ]

            true = batch_y[
                :,
                -args.pred_len:,
                :
            ]

            loss = F.mse_loss(
                outputs,
                true,
            )

            loss.backward()
            optimizer.step()

            losses.append(
                float(
                    loss.item()
                )
            )

            del (
                batch_x,
                batch_y,
                batch_x_mark,
                batch_y_mark,
                dec_inp,
                outputs,
                true,
                loss,
            )

        train_mse = float(
            np.mean(losses)
        )

        val = evaluate_loader(
            model,
            val_loader,
            args,
        )

        val_mse = val[
            "BatchAverageMSE"
        ]

        if (
            best_state is None
            or val_mse <= best_val
        ):
            best_val = val_mse
            best_epoch = epoch

            best_state = {
                k:
                    v.detach()
                    .cpu()
                    .clone()
                for k, v
                in model.state_dict().items()
            }

            wait = 0
        else:
            wait += 1

        # Match official type1 scheduler.
        official_adjust_lr(
            optimizer,
            epoch,
            args,
        )

        current_lr = optimizer.param_groups[
            0
        ][
            "lr"
        ]

        history.append({
            "Epoch": epoch,
            "TrainMSE": train_mse,
            "ValBatchAverageMSE": val_mse,
            "ValGlobalMSE": val["MSE"],
            "ValMAE": val["MAE"],
            "BestValMSE": best_val,
            "BestEpoch": best_epoch,
            "LR": current_lr,
            "Seconds": time.time() - t0,
        })

        pd.DataFrame(
            history
        ).to_csv(
            history_path(
                name,
                horizon,
            ),
            index=False,
        )

        print(
            f"{name:11s} H={horizon:3d} "
            f"ep={epoch:02d}/{args.train_epochs} | "
            f"train={train_mse:.6f} | "
            f"val={val_mse:.6f} | "
            f"best={best_val:.6f}@{best_epoch} | "
            f"lr={current_lr:.3e} | "
            f"wait={wait}/{args.patience}"
        )

        if wait >= args.patience:
            print(
                f"Early stopping: {name} H={horizon}"
            )
            break

    if best_state is None:
        raise RuntimeError(
            f"{name} H={horizon}: no best checkpoint."
        )

    model.load_state_dict(
        best_state
    )

    model.eval()

    ckpt = {
        "Dataset": name,
        "Horizon": int(horizon),
        "SeqLen": SEQ_LEN,
        "Seed": SEED,
        "BestEpoch": int(best_epoch),
        "BestValMSE": float(best_val),
        "Recipe": RECIPES[name],
        "OfficialRepoCommit": commit,
        "StateDict": best_state,
    }

    torch.save(
        ckpt,
        path,
    )

    del (
        train_data,
        train_loader,
        val_data,
        val_loader,
    )

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return (
        model,
        args,
        ckpt,
    )


In [ ]:

SUMMARY_PATH = ROOT / "summary.csv"

existing = (
    pd.read_csv(
        SUMMARY_PATH
    )
    if (
        RESUME
        and SUMMARY_PATH.exists()
    )
    else pd.DataFrame()
)

result_rows = (
    existing.to_dict("records")
    if len(existing)
    else []
)


def already_done(
    name,
    horizon,
):
    if not len(existing):
        return False

    return bool(
        (
            (
                existing["Dataset"]
                == name
            )
            & (
                existing["Horizon"]
                == horizon
            )
        ).any()
    )


for name, horizon in TASKS:
    if already_done(
        name,
        horizon,
    ):
        print(
            f"SKIP completed: "
            f"{name} H={horizon}"
        )
        continue

    print(
        "\n"
        + "=" * 140
    )

    print(
        f"OFFICIAL iTransformer | "
        f"{name} | L=96 | H={horizon}"
    )

    print(
        "=" * 140
    )

    t0 = time.time()

    model, args, ckpt = train_or_load(
        name,
        horizon,
    )

    test_data, test_loader = official_data_provider(
        args,
        "test",
    )

    test = evaluate_loader(
        model,
        test_loader,
        args,
    )

    ref_mse = REFERENCE_MSE[
        (
            name,
            horizon,
        )
    ]

    row = {
        "Dataset": name,
        "Horizon": horizon,
        "SeqLen": SEQ_LEN,
        "Channels": RECIPES[name]["enc_in"],
        "BestEpoch": ckpt["BestEpoch"],
        "BestValMSE": ckpt["BestValMSE"],
        "Test_MSE": test["MSE"],
        "Test_MAE": test["MAE"],
        "TestWindows": test["Windows"],
        "Reference_MSE": ref_mse,
        "AbsDiffVsReference": (
            test["MSE"]
            - ref_mse
        ),
        "RelDiffVsReference_pct": (
            100.0
            * (
                test["MSE"]
                - ref_mse
            )
            / ref_mse
        ),
        "RuntimeMinutes": (
            time.time()
            - t0
        )
        / 60.0,
        "Checkpoint": str(
            checkpoint_path(
                name,
                horizon,
            )
        ),
        "OfficialRepoCommit": commit,
    }

    result_rows.append(
        row
    )

    pd.DataFrame(
        result_rows
    ).to_csv(
        SUMMARY_PATH,
        index=False,
    )

    display(
        pd.DataFrame(
            [row]
        )[
            [
                "Dataset",
                "Horizon",
                "BestEpoch",
                "Test_MSE",
                "Test_MAE",
                "Reference_MSE",
                "RelDiffVsReference_pct",
                "RuntimeMinutes",
            ]
        ]
    )

    del (
        model,
        args,
        ckpt,
        test_data,
        test_loader,
    )

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


summary_df = (
    pd.DataFrame(
        result_rows
    )
    .sort_values(
        [
            "Dataset",
            "Horizon",
        ]
    )
    .reset_index(
        drop=True
    )
)

display(
    summary_df
)


In [ ]:

compact = summary_df[
    [
        "Dataset",
        "Horizon",
        "Test_MSE",
        "Test_MAE",
        "Reference_MSE",
        "RelDiffVsReference_pct",
        "BestEpoch",
        "RuntimeMinutes",
    ]
].copy()

compact["FidelityBand"] = pd.cut(
    compact["RelDiffVsReference_pct"].abs(),
    bins=[
        -np.inf,
        3.0,
        7.5,
        np.inf,
    ],
    labels=[
        "Close (<=3%)",
        "Moderate (3-7.5%)",
        "Large (>7.5%)",
    ],
)

display(
    compact
)

compact.to_csv(
    ROOT / "compact_fidelity_table.csv",
    index=False,
)


In [ ]:

if len(summary_df):
    runtime_summary = (
        summary_df
        .groupby(
            "Dataset",
            as_index=False,
        )
        .agg(
            Conditions=(
                "Horizon",
                "size",
            ),
            TotalRuntimeMinutes=(
                "RuntimeMinutes",
                "sum",
            ),
            MeanRuntimeMinutes=(
                "RuntimeMinutes",
                "mean",
            ),
        )
    )

    runtime_summary[
        "TotalRuntimeHours"
    ] = (
        runtime_summary[
            "TotalRuntimeMinutes"
        ]
        / 60.0
    )

    display(
        runtime_summary
    )

    print(
        "Completed total runtime (hours):",
        summary_df[
            "RuntimeMinutes"
        ].sum()
        / 60.0,
    )


In [ ]:

print(
    "Experiment root:",
    ROOT,
)

for p in sorted(
    ROOT.rglob("*")
):
    if p.is_file():
        print(
            p.relative_to(
                ROOT
            )
        )
